In [1]:
import sys, os
sys.path.append('/content/drive/MyDrive/develop/Projects/quant-dev-git/src')
os.chdir('/content/drive/MyDrive/develop/Projects/quant-dev-git')
print(os.getcwd())

/content/drive/MyDrive/develop/Projects/quant-dev-git


In [9]:
from quant_dev.data.manager import DataManager
# 1. 準備數據
dm = DataManager()
ticker = "AAPL"
df = dm.get_or_fetch(ticker, timeframe="1d", days=500, force_download=True)
print(f"Success! Data shape: {df.shape}")

Success! Data shape: (342, 5)


In [10]:
from quant_dev.backtest.strategy import Strategy, StrategyConfig
import pandas as pd

# 2. 建立一個簡單嘅信號 (例如: 當收市價高於50日移動平均線就買入)
df['sma_50'] = df['Close'].rolling(window=50).mean()
df['signal'] = 0
df.loc[df['Close'] > df['sma_50'], 'signal'] = 1
df.loc[df['Close'] < df['sma_50'], 'signal'] = -1

# 3. 設定策略配置
config = StrategyConfig(
    ticker=ticker,
    timeframe="1d",
    direction="buy",
    mode="normal",
    entry_order_type="market",  # 用市價單簡化測試
    exit_order_type="market",
    gap_entry="open",
    gap_exit="open",
    initial_capital=100000.0
)

# 4. 執行回測
print("Running Strategy backtest...")
strategy = Strategy(config, df)
strategy.add_signal(df['signal'])  # 加入我哋自訂嘅信號
strategy.run()

# 5. 查看結果
metrics = strategy.get_performance_metrics()
print("\n策略績效:")
for key, value in metrics.items():
    print(f"{key}: {value}")

# 顯示部分交易記錄
print("\n頭5筆交易記錄:")
print(strategy.df[['Close', 'position', 'entry', 'exit']].head(10))


Running Strategy backtest...

策略績效:
Total Return (%): 30.03
Annual Return (%): 21.35
Sharpe Ratio: 1.08
Max Drawdown (%): 12.6

頭5筆交易記錄:
                 Close  position  entry  exit
Date                                         
2025-03-24  219.569855         0    NaN   NaN
2025-03-25  222.573975         0    NaN   NaN
2025-03-26  220.365631         0    NaN   NaN
2025-03-27  222.673462         0    NaN   NaN
2025-03-28  216.754700         0    NaN   NaN
2025-03-31  220.962494         0    NaN   NaN
2025-04-01  222.016907         0    NaN   NaN
2025-04-02  222.713242         0    NaN   NaN
2025-04-03  202.122025         0    NaN   NaN
2025-04-04  187.389877         0    NaN   NaN
